In [34]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/Users/putri/Documents/!! BI NY/Data Bloomberg Fiscal Impulse/!! Fiscal Impulse/(Exponential) dataset_fiscal_impulse_updated.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst 3 rows:")
print(df.head(3))

Dataset shape: (124, 13)
Columns: ['Date', 'gdp_potential_brazil', 'gdp_potential_canada', 'gdp_potential_mexico', 'pb_brazil', 'pb_canada', 'pb_mexico', 'output_gap_pct_brazil', 'output_gap_pct_canada', 'output_gap_pct_mexico', 'semi_elasticity_brazil', 'semi_elasticity_canada', 'semi_elasticity_mexico']

First 3 rows:
         Date  gdp_potential_brazil  gdp_potential_canada  \
0  2026-12-31         352853.951123         624296.268820   
1  2026-09-30         351290.347069         621251.350887   
2  2026-06-30         349498.575640         619167.421632   

   gdp_potential_mexico  pb_brazil  pb_canada  pb_mexico  \
0          6.529478e+06      44.82       2.03       3.66   
1          6.507311e+06      -7.25       2.62      30.35   
2          6.483070e+06      23.84       4.32      63.74   

   output_gap_pct_brazil  output_gap_pct_canada  output_gap_pct_mexico  \
0              -0.751464               0.577525              -0.232199   
1              -0.558108              -0.040

In [35]:
# Show raw column names
df.columns.tolist()

['Date',
 'gdp_potential_brazil',
 'gdp_potential_canada',
 'gdp_potential_mexico',
 'pb_brazil',
 'pb_canada',
 'pb_mexico',
 'output_gap_pct_brazil',
 'output_gap_pct_canada',
 'output_gap_pct_mexico',
 'semi_elasticity_brazil',
 'semi_elasticity_canada',
 'semi_elasticity_mexico']

In [36]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['Date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of observations: {len(df)}")


Date range: 1996-03-31 00:00:00 to 2026-12-31 00:00:00
Number of observations: 124


In [37]:
# Formula
print("\n\nFormula")
print("-" * 40)
print("Formula: CAPB = (PB - ε × OG × GDP) / GDP")
print("Where:")
print("  PB = Primary Balance (% of GDP)")
print("  ε  = Semi-elasticity (country-specific)")
print("  OG = Output Gap (% of potential GDP)")
print("  GDP = Potential GDP")
print("\nFiscal Impulse = -(CAPB_t - CAPB_t-1)")



Formula
----------------------------------------
Formula: CAPB = (PB - ε × OG × GDP) / GDP
Where:
  PB = Primary Balance (% of GDP)
  ε  = Semi-elasticity (country-specific)
  OG = Output Gap (% of potential GDP)
  GDP = Potential GDP

Fiscal Impulse = -(CAPB_t - CAPB_t-1)


In [ ]:
# df['pb_canada'] = (df['pb_canada'] / df['gdp_potential_canada']) * 100

In [39]:
# CAPB Calculation
print("\n\nCalculate CAPB (Cyclically Adjusted Primary Balance)")
print("-" * 40)

def calculate_capb(country):
    """Calculate CAPB for each country"""
    
    print(f"\nCalculating CAPB for {country.upper()}:")
    
    # Extract data columns
    pb_col = f'pb_{country}'
    gdp_col = f'gdp_potential_{country}'
    og_col = f'output_gap_pct_{country}'
    epsilon_col = f'semi_elasticity_{country}'
    
    idx = 0  # First row
    pb = df[pb_col].iloc[idx]
    gdp = df[gdp_col].iloc[idx]
    og = df[og_col].iloc[idx]
    epsilon = df[epsilon_col].iloc[idx]
    
    print(f"  Sample calculation for {df['date'].iloc[idx].strftime('%Y-%m-%d')}:")
    print(f"    PB = {pb}%")
    print(f"    GDP = {gdp:,.0f}")
    print(f"    Output Gap = {og:.3f}%")
    print(f"    Semi-elasticity (ε) = {epsilon}")
    
    # Calculation
    og_decimal = og * 100  # Convert percentage to decimal
    cyclical_component = epsilon * og_decimal * gdp
    adjusted_pb = pb - cyclical_component
    capb = adjusted_pb / gdp
    
    print(f"    Step 1: Convert OG to decimal = {og_decimal:.6f}")
    print(f"    Step 2: Cyclical component = ε × OG × GDP = {epsilon} × {og_decimal:.6f} × {gdp:,.0f} = {cyclical_component:.6f}")
    print(f"    Step 3: Adjusted PB = PB - Cyclical = {pb} - {cyclical_component:.6f} = {adjusted_pb:.6f}")
    print(f"    Step 4: CAPB = Adjusted PB / GDP = {adjusted_pb:.6f} / {gdp:,.0f} = {capb:.8f}")
    print(f"    Final CAPB = {capb * 100:.6f}% of GDP")
    
    # Calculate CAPB for all periods
    df[f'output_gap_decimal_{country}'] = df[og_col] / 100
    df[f'cyclical_component_{country}'] = df[epsilon_col] * df[f'output_gap_decimal_{country}'] * df[gdp_col]
    df[f'adjusted_pb_{country}'] = df[pb_col] - df[f'cyclical_component_{country}']
    df[f'capb_{country}'] = df[f'adjusted_pb_{country}'] / df[gdp_col]
    
    return df[f'capb_{country}']

# Calculate CAPB for all countries
countries = ['brazil', 'canada', 'mexico']
for country in countries:
    calculate_capb(country)

# Fiscal Impulse Calculation
def calculate_fiscal_impulse(country):
    capb_col = f'capb_{country}'
    df[f'capb_change_{country}'] = df[capb_col].diff()
    df[f'fiscal_impulse_{country}'] = -df[f'capb_change_{country}'] * 100

for country in countries:
    calculate_fiscal_impulse(country)





Calculate CAPB (Cyclically Adjusted Primary Balance)
----------------------------------------

Calculating CAPB for BRAZIL:
  Sample calculation for 1996-03-31:
    PB = 3.778%
    GDP = 175,461
    Output Gap = 0.000%
    Semi-elasticity (ε) = 0.5
    Step 1: Convert OG to decimal = 0.000000
    Step 2: Cyclical component = ε × OG × GDP = 0.5 × 0.000000 × 175,461 = 0.000000
    Step 3: Adjusted PB = PB - Cyclical = 3.778 - 0.000000 = 3.778000
    Step 4: CAPB = Adjusted PB / GDP = 3.778000 / 175,461 = 0.00002153
    Final CAPB = 0.002153% of GDP

Calculating CAPB for CANADA:
  Sample calculation for 1996-03-31:
    PB = 0.4716%
    GDP = 317,339
    Output Gap = -0.075%
    Semi-elasticity (ε) = 0.55
    Step 1: Convert OG to decimal = -7.486684
    Step 2: Cyclical component = ε × OG × GDP = 0.55 × -7.486684 × 317,339 = -1306697.526338
    Step 3: Adjusted PB = PB - Cyclical = 0.4716 - -1306697.526338 = 1306697.997938
    Step 4: CAPB = Adjusted PB / GDP = 1306697.997938 / 317,339 

In [40]:
# Calculate Fiscal Impulse
print("\n\nFiscal Impulse")
print("-" * 40)
print("Fiscal Impulse = -(CAPB_t - CAPB_t-1)")

def calculate_fiscal_impulse_step_by_step(country):
    """Calculate fiscal impulse"""
    
    print(f"\nFiscal Impulse for {country.upper()}:")
    
    capb_col = f'capb_{country}'
    
    # Sample
    if len(df) > 1:
        idx = 1  
        capb_current = df[capb_col].iloc[idx]
        capb_previous = df[capb_col].iloc[idx-1]
        
        print(f"  Sample calculation for {df['date'].iloc[idx].strftime('%Y-%m-%d')}:")
        print(f"    CAPB_current = {capb_current * 100:.6f}% (period t)")
        print(f"    CAPB_previous = {capb_previous * 100:.6f}% (period t-1)")
        print(f"    Change in CAPB = {capb_current * 100:.6f} - {capb_previous * 100:.6f} = {(capb_current - capb_previous) * 100:.6f}")
        print(f"    Fiscal Impulse = -({(capb_current - capb_previous) * 100:.6f}) = {-(capb_current - capb_previous) * 100:.6f}%")
    
    # Fiscal impulse for all periods
    df[f'capb_change_{country}'] = df[capb_col].diff()
    df[f'fiscal_impulse_{country}'] = -df[f'capb_change_{country}'] * 100
    
    return df[f'fiscal_impulse_{country}']

# Fiscal impulse for all countries
for country in countries:
    calculate_fiscal_impulse_step_by_step(country)

# Fiscal Impulse Results
print("\n\nFiscal Impulse Results")
print("-" * 40)

# Results summary
impulse_cols = ['date'] + [f'fiscal_impulse_{country}' for country in countries]
impulse_summary = df[impulse_cols].dropna()  

print("Fiscal Impulse (% of GDP) - First 10 periods:")
print(impulse_summary.head(10).round(6))

print("\nFiscal Impulse (% of GDP) - Last 5 periods:")
print(impulse_summary.tail(10).round(6))




Fiscal Impulse
----------------------------------------
Fiscal Impulse = -(CAPB_t - CAPB_t-1)

Fiscal Impulse for BRAZIL:
  Sample calculation for 1996-06-30:
    CAPB_current = -0.000459% (period t)
    CAPB_previous = 0.002153% (period t-1)
    Change in CAPB = -0.000459 - 0.002153 = -0.002612
    Fiscal Impulse = -(-0.002612) = 0.002612%

Fiscal Impulse for CANADA:
  Sample calculation for 1996-06-30:
    CAPB_current = -0.008626% (period t)
    CAPB_previous = 0.041325% (period t-1)
    Change in CAPB = -0.008626 - 0.041325 = -0.049952
    Fiscal Impulse = -(-0.049952) = 0.049952%

Fiscal Impulse for MEXICO:
  Sample calculation for 1996-06-30:
    CAPB_current = -0.573755% (period t)
    CAPB_previous = -0.419130% (period t-1)
    Change in CAPB = -0.573755 - -0.419130 = -0.154625
    Fiscal Impulse = -(-0.154625) = 0.154625%


Fiscal Impulse Results
----------------------------------------
Fiscal Impulse (% of GDP) - First 10 periods:
         date  fiscal_impulse_brazil  fisca

In [41]:
## STANCE MAPPING

# 1. Set up thresholds and classify historical scenario stances per country
stance_labels = [
    "Strong Expansionary", "Modest Expansionary", "Slightly Expansionary", "Neutral",
    "Slightly Contractionary", "Modest Contractionary", "Strong Contractionary"
]

for country in ['canada', 'brazil', 'mexico']:
    vals = df[f'fiscal_impulse_{country}'].dropna()
    # Use historical percentiles as cutoff
    strong_exp = np.percentile(vals, 90)
    modest_exp = np.percentile(vals, 75)
    slight_exp = np.percentile(vals, 60)
    neutral = np.percentile(np.abs(vals), 20)
    slight_con = np.percentile(vals, 40)
    modest_con = np.percentile(vals, 25)
    strong_con = np.percentile(vals, 10)
    def classify(val):
        if val >= strong_exp: return "Strong Expansionary"
        elif val >= modest_exp: return "Modest Expansionary"
        elif val >= slight_exp: return "Slightly Expansionary"
        elif abs(val) < neutral: return "Neutral"
        elif val <= strong_con: return "Strong Contractionary"
        elif val <= modest_con: return "Modest Contractionary"
        elif val <= slight_con: return "Slightly Contractionary"
        return "Neutral"
    df[f'scenario_stance_{country}'] = df[f'fiscal_impulse_{country}'].apply(classify)


In [42]:

# 2. Calculate country-specific mean (or median) for each scenario stance
stance_means = {}
for country in ['canada', 'brazil', 'mexico']:
    stance_means[country] = (
        df.groupby(f'scenario_stance_{country}')[f'fiscal_impulse_{country}']
        .mean().to_dict()
    )
    print(f"Stance means for {country.upper()}: ", stance_means[country])

Stance means for CANADA:  {'Modest Contractionary': -0.2640921258943145, 'Modest Expansionary': 0.2897311758733005, 'Neutral': 0.015867769241308113, 'Slightly Contractionary': -0.13713279807436923, 'Slightly Expansionary': 0.13520322505387142, 'Strong Contractionary': -1.090458196879091, 'Strong Expansionary': 1.0193945196271488}
Stance means for BRAZIL:  {'Modest Contractionary': -0.34389518280766207, 'Modest Expansionary': 0.3125294501910584, 'Neutral': 0.00910204950104625, 'Slightly Contractionary': -0.148412911764001, 'Slightly Expansionary': 0.18815306705160045, 'Strong Contractionary': -1.085807357377863, 'Strong Expansionary': 1.002761080565676}
Stance means for MEXICO:  {'Modest Contractionary': -0.19238437205134107, 'Modest Expansionary': 0.21655522779857053, 'Neutral': -0.03280626208140081, 'Slightly Contractionary': -0.11200613367197919, 'Slightly Expansionary': 0.08112370283619257, 'Strong Contractionary': -0.7839022402154388, 'Strong Expansionary': 0.8185020144674358}


In [43]:
fi_cols = [f'fiscal_impulse_{country}' for country in ['canada', 'brazil', 'mexico']]
comparison = df.melt(
    id_vars='date',
    value_vars=fi_cols,
    var_name='country',
    value_name='fiscal_impulse'
)
comparison['country'] = comparison['country'].str.replace('fiscal_impulse_', '', regex=False).str.upper()

In [44]:
# Set scenario stances for 2025Q2-2026Q4 for CANADA, BRAZIL, MEXICO
manual_stance_periods = [
    # CANADA
    ["2025-06-30", "CANADA", "Modest Expansionary"],
    ["2025-09-30", "CANADA", "Modest Expansionary"],
    ["2025-12-31", "CANADA", "Modest Expansionary"],
    ["2026-03-31", "CANADA", "Modest Expansionary"],
    ["2026-06-30", "CANADA", "Modest Expansionary"],
    ["2026-09-30", "CANADA", "Slightly Expansionary"],
    ["2026-12-31", "CANADA", "Neutral"],
    # BRAZIL
    ["2025-06-30", "BRAZIL", "Modest Contractionary"],
    ["2025-09-30", "BRAZIL", "Modest Contractionary"],
    ["2025-12-31", "BRAZIL", "Modest Contractionary"],
    ["2026-03-31", "BRAZIL", "Slightly Contractionary"],
    ["2026-06-30", "BRAZIL", "Slightly Contractionary"],
    ["2026-09-30", "BRAZIL", "Slightly Contractionary"],
    ["2026-12-31", "BRAZIL", "Neutral"],
    # MEXICO
    ["2025-06-30", "MEXICO", "Slightly Contractionary"],
    ["2025-09-30", "MEXICO", "Modest Contractionary"],
    ["2025-12-31", "MEXICO", "Modest Contractionary"],
    ["2026-03-31", "MEXICO", "Modest Contractionary"],
    ["2026-06-30", "MEXICO", "Slightly Contractionary"],
    ["2026-09-30", "MEXICO", "Slightly Contractionary"],
    ["2026-12-31", "MEXICO", "Slightly Contractionary"],
]
scenario_manual = pd.DataFrame(manual_stance_periods, columns=["date", "country", "scenario_stance"])
scenario_manual["date"] = pd.to_datetime(scenario_manual["date"])


In [28]:
comparison = comparison.merge(
    scenario_manual,
    how="left",
    on=["date", "country"]
)


In [29]:
print(comparison.tail(15))
print(comparison.head())


          date country  fiscal_impulse                   scenario_stance_x  \
357 2023-06-30  MEXICO       -0.075232                                 NaN   
358 2023-09-30  MEXICO       -0.148709                                 NaN   
359 2023-12-31  MEXICO       -0.162507                                 NaN   
360 2024-03-31  MEXICO       -0.238442                                 NaN   
361 2024-06-30  MEXICO       -0.158471                                 NaN   
362 2024-09-30  MEXICO        0.035652                                 NaN   
363 2024-12-31  MEXICO       -0.348253                                 NaN   
364 2025-03-31  MEXICO       -0.067113                                 NaN   
365 2025-06-30  MEXICO       -0.168880             Slightly Contractionary   
366 2025-09-30  MEXICO        0.178038  Neutral or Slightly Contractionary   
367 2025-12-31  MEXICO       -0.079191  Neutral or Slightly Contractionary   
368 2026-03-31  MEXICO        0.031870  Neutral or Slightly Cont

In [30]:
def override_with_scenario(row):
    country = row['country'].lower()
    stance = row.get('scenario_stance')
    if stance and stance in stance_means[country]:
        return stance_means[country][stance]
    else:
        return row['fiscal_impulse']

# Only override for periods starting from 2025Q2. Earlier periods use factual impulse.
comparison['fiscal_impulse_adjusted'] = comparison.apply(
    lambda row: override_with_scenario(row) if row['date'] >= pd.Timestamp("2025-06-30") else row['fiscal_impulse'],
    axis=1
)


In [31]:
def override_with_scenario(row):
    country = row['country'].lower()
    stance = row.get('scenario_stance')
    if pd.notna(stance) and stance in stance_means[country]:
        return stance_means[country][stance]
    else:
        return row['fiscal_impulse']

comparison['fiscal_impulse_final'] = comparison.apply(override_with_scenario, axis=1)


In [33]:
print(comparison.columns)
print(comparison[['date', 'country', 'fiscal_impulse', 'fiscal_impulse_final', 'scenario_stance']].tail(20))


Index(['date', 'country', 'fiscal_impulse', 'scenario_stance_x',
       'scenario_stance_y', 'fiscal_impulse_adjusted', 'fiscal_impulse_final',
       'scenario_stance'],
      dtype='object')
          date country  fiscal_impulse  fiscal_impulse_final  \
352 2022-03-31  MEXICO        0.288817              0.288817   
353 2022-06-30  MEXICO        0.122907              0.122907   
354 2022-09-30  MEXICO       -0.058345             -0.058345   
355 2022-12-31  MEXICO        0.112334              0.112334   
356 2023-03-31  MEXICO       -0.077149             -0.077149   
357 2023-06-30  MEXICO       -0.075232             -0.075232   
358 2023-09-30  MEXICO       -0.148709             -0.148709   
359 2023-12-31  MEXICO       -0.162507             -0.162507   
360 2024-03-31  MEXICO       -0.238442             -0.238442   
361 2024-06-30  MEXICO       -0.158471             -0.158471   
362 2024-09-30  MEXICO        0.035652              0.035652   
363 2024-12-31  MEXICO       -0.348253 

In [7]:
# Create table
impulse_cols = ['date'] + [f'fiscal_impulse_{country}' for country in countries]
impulse_summary = df[impulse_cols].dropna()

# Round values
impulse_summary = impulse_summary.round(6)

# Print first 20 rows
print("Fiscal Impulse (% of GDP) — First 20 periods:")
print(impulse_summary.head(20).to_string(index=False))

# Print last 10 rows
print("\nFiscal Impulse (% of GDP) — Last 20 periods:")
print(impulse_summary.tail(20).to_string(index=False))


Fiscal Impulse (% of GDP) — First 20 periods:
      date  fiscal_impulse_brazil  fiscal_impulse_canada  fiscal_impulse_mexico
1996-06-30               0.002612               0.049952               0.154625
1996-09-30              -0.000840               0.097199               0.200354
1996-12-31              -0.433896               0.056147               0.536029
1997-03-31               0.179472               0.282539              -0.288059
1997-06-30              -0.035923               0.124357               0.246733
1997-09-30               0.196095               0.115791               0.105497
1997-12-31              -0.001413              -0.016582               0.237494
1998-03-31              -0.870100               0.170736              -0.160741
1998-06-30               0.644345              -0.478089              -0.353600
1998-09-30              -0.135084              -0.007087              -0.350489
1998-12-31              -0.476399               0.183091              -0.3

In [8]:
# Output path
output_path = "/Users/putri/Documents/!! BI NY/Data Bloomberg Fiscal Impulse/!! Fiscal Impulse/(Exponential) fiscal_impulse_results.xlsx"

# Safe df
df.to_excel(output_path, index=False)

print(f"Fiscal impulse results saved to: {output_path}")


Fiscal impulse results saved to: /Users/putri/Documents/!! BI NY/Data Bloomberg Fiscal Impulse/!! Fiscal Impulse/(Exponential) fiscal_impulse_results.xlsx


In [9]:
# Adjust Format
fi_cols = [f"fiscal_impulse_{country}" for country in countries]
comparison = df.melt(id_vars="date", value_vars=fi_cols,
                     var_name="country", value_name="fiscal_impulse")

# Country Name
comparison["country"] = comparison["country"].str.replace("fiscal_impulse_", "", regex=False).str.upper()

# Categorize
def classify_impulse(val):
    if pd.isna(val):
        return None
    if val > 0:
        return "Expansionary"
    elif val < 0:
        return "Contractionary"
    else:
        return "Neutral"

comparison["stance"] = comparison["fiscal_impulse"].apply(classify_impulse)

# Save
with pd.ExcelWriter(output_path) as writer:
    df.to_excel(writer, sheet_name="Full Data", index=False)
    comparison.to_excel(writer, sheet_name="Comparison Table", index=False)

print(f"Saved both full data and comparison table to: {output_path}")


Saved both full data and comparison table to: /Users/putri/Documents/!! BI NY/Data Bloomberg Fiscal Impulse/!! Fiscal Impulse/(Exponential) fiscal_impulse_results.xlsx


In [10]:
!pip install nbconvert
!jupyter nbconvert --to html "(Exponential) Fiscal Impulse Brazil, Canada, Mexico (1996-2005).ipynb"


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
[NbConvertApp] Converting notebook (Exponential) Fiscal Impulse Brazil, Canada, Mexico (1996-2005).ipynb to html
[NbConvertApp] Writing 331936 bytes to (Exponential) Fiscal Impulse Brazil, Canada, Mexico (1996-2005).html
